# NPORT-P 原 XML 报告日期字段零网络核查

在同一已封存来源中区分 `repPdDate`、`repPdEnd` 和 `dateSigned`；只读字段名、身份和投资条目数，不算市值。


In [1]:
import json,hashlib,xml.etree.ElementTree as ET
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_2019_nport_source_specimen_20261023';plan=json.loads((card/'schema_semantics_repair_plan.json').read_text());assert plan['frozen']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(card/'result.json')==plan['parent_result_sha256']
raw=root/plan['raw_path'];assert sha(raw)==plan['raw_sha256']
xml=ET.fromstring(raw.read_bytes());local=lambda t:t.rsplit('}',1)[-1]
def texts(tag):return sorted({(e.text or '').strip() for e in xml.iter() if local(e.tag)==tag and (e.text or '').strip()})
identity={'cik':texts('cik'),'seriesId':texts('seriesId'),'classId':texts('classId'),'repPdDate':texts('repPdDate'),'repPdEnd':texts('repPdEnd'),'dateSigned':texts('dateSigned')}
assert identity['cik']==['0001100663'] and identity['seriesId']==['S000004354'] and identity['classId']==['C000012084']
assert identity['repPdDate']==['2019-09-30']
source=json.loads((root/'research/stage1_soxx_2019_2022_nport_filing_metadata_20261023/metadata_result.json').read_text())
parent=next(x for x in source['rows'] if x['accession']=='0001752724-19-177706')
assert parent['report_period']==identity['repPdDate'][0]
rows=[e for e in xml.iter() if local(e.tag)=='invstOrSec'];assert len(rows)==34
fields=[local(e.tag) for e in rows[0].iter()][1:]
assert {'name','cusip','balance','valUSD','pctVal'}.issubset(fields)
result={'status':'one_original_NPORT_XML_identity_report_date_and_holdings_schema_qualified','source_raw_sha256':plan['raw_sha256'],'source_NAS_archive_sha256':plan['nas_archive_sha256'],'source_NAS_scoped_restore_verified':True,'source_index_accession':parent['accession'],'source_index_accepted_display':parent['accepted_display'],'identity_fields':identity,'original_investment_rows':len(rows),'first_investment_row_field_names':fields,'holdings_values_parsed':0,'full_training_coverage_qualified':False,'acceptance_timezone_qualified':False,'new_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'schema_semantics_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','original_investment_rows','identity_fields','new_HTTP_GETs')})


{'status': 'one_original_NPORT_XML_identity_report_date_and_holdings_schema_qualified', 'original_investment_rows': 34, 'identity_fields': {'cik': ['0001100663'], 'seriesId': ['S000004354'], 'classId': ['C000012084'], 'repPdDate': ['2019-09-30'], 'repPdEnd': ['2020-03-31'], 'dateSigned': ['2019-10-29']}, 'new_HTTP_GETs': 0}


## 中文结论

从相同已封存 XML 零网络确认 `repPdDate=2019-09-30` 与 SEC filing index 一致；`repPdEnd=2020-03-31`、`dateSigned=2019-10-29` 分开保留，不冒充首次公开时点。34 个投资条目字段含 name/cusip/balance/valUSD/pctVal。单一样本通过来源身份和结构，不证明全期数值或超额。
